La variable objetivo seleccionada para el modelado es Factura_Promedio_COP.

Las variables candidatas para el modelo son:

- Empresa
- Segmento
- Mes
- Año

In [46]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [47]:
df = pd.read_csv("sui_factura_promedio_consolidado.csv")

In [48]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.isnull().sum()

(5809, 6)
['Empresa', 'Anio', 'Mes', 'Periodo', 'Segmento', 'Factura_Promedio_COP']
Empresa                     str
Anio                      int64
Mes                       int64
Periodo                     str
Segmento                    str
Factura_Promedio_COP    float64
dtype: object


Empresa                 0
Anio                    0
Mes                     0
Periodo                 0
Segmento                0
Factura_Promedio_COP    0
dtype: int64

In [49]:
columnas = ['Anio', 'Mes', 'Segmento', 'Factura_Promedio_COP']

df = df[columnas]

Normalizar nombres: minúsculas, sin espacios

In [50]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [51]:
print(df)

      anio  mes              segmento  factura_promedio_cop
0     2025    1             Estrato 1          2.730945e+05
1     2025    1             Estrato 1          2.002508e+05
2     2025    1             Estrato 1          8.711506e+04
3     2025    1             Estrato 1          1.191730e+05
4     2025    1             Estrato 1          7.251058e+04
...    ...  ...                   ...                   ...
5804  2026    3  Total No Residencial          8.367119e+06
5805  2026    3  Total No Residencial          3.693736e+06
5806  2026    3  Total No Residencial          2.067820e+07
5807  2026    3  Total No Residencial          2.784656e+07
5808  2026    3  Total No Residencial          5.552995e+08

[5809 rows x 4 columns]


In [52]:
df.dtypes

anio                      int64
mes                       int64
segmento                    str
factura_promedio_cop    float64
dtype: object

In [53]:
df["factura_promedio_cop"] = (
    df["factura_promedio_cop"]
    .where(
        df["factura_promedio_cop"] >= 0,
        np.nan
    )
)

segmentos_totales = ["Total Residencial", "Total No Residencial"]
df = df[~df["segmento"].isin(segmentos_totales)].copy()
df = df.dropna(subset=["factura_promedio_cop"]).copy()

In [54]:
print("Filas después de excluir totales:", len(df))
print("¿Quedan totales?:", df["segmento"].isin(["Total Residencial", "Total No Residencial"]).any())
print("Segmentos:", sorted(df["segmento"].unique().tolist()))

Filas después de excluir totales: 4633
¿Quedan totales?: False
Segmentos: ['Comercial', 'Estrato 1', 'Estrato 2', 'Estrato 3', 'Estrato 4', 'Estrato 5', 'Estrato 6', 'Industrial', 'Oficial', 'Otros']


In [55]:
columns_category = df.select_dtypes(include=['object', 'string', 'category']).columns
print(columns_category)

Index(['segmento'], dtype='str')


In [56]:
df_model = pd.get_dummies(
    df,
    columns=columns_category,
    drop_first=True
)

In [57]:
df_model.dtypes.value_counts()

bool       9
int64      2
float64    1
Name: count, dtype: int64

In [58]:
df_model.dtypes

anio                      int64
mes                       int64
factura_promedio_cop    float64
segmento_Estrato 1         bool
segmento_Estrato 2         bool
segmento_Estrato 3         bool
segmento_Estrato 4         bool
segmento_Estrato 5         bool
segmento_Estrato 6         bool
segmento_Industrial        bool
segmento_Oficial           bool
segmento_Otros             bool
dtype: object

In [59]:
df_model['factura_promedio_cop'].describe()

count    4.633000e+03
mean     2.770042e+07
std      1.390240e+08
min      0.000000e+00
25%      1.169187e+05
50%      3.722959e+05
75%      3.195020e+06
max      1.969094e+09
Name: factura_promedio_cop, dtype: float64

In [60]:
print((df_model["factura_promedio_cop"] < 0).sum())

0


In [61]:
df_model['log_factura'] = np.log1p(df_model['factura_promedio_cop'])

60% train, 20% val, 20% test

In [62]:
df_train_full, df_test = train_test_split(df_model, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

In [63]:
print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 2779, Val: 927, Test: 927


In [64]:
df_train.dtypes

anio                      int64
mes                       int64
factura_promedio_cop    float64
segmento_Estrato 1         bool
segmento_Estrato 2         bool
segmento_Estrato 3         bool
segmento_Estrato 4         bool
segmento_Estrato 5         bool
segmento_Estrato 6         bool
segmento_Industrial        bool
segmento_Oficial           bool
segmento_Otros             bool
log_factura             float64
dtype: object

In [65]:
y_train = df_train['log_factura'].values
y_val = df_val['log_factura'].values
y_test = df_test['log_factura'].values

In [66]:
del df_train['log_factura'], df_train['factura_promedio_cop']
del df_val['log_factura'], df_val['factura_promedio_cop']
del df_test['log_factura'], df_test['factura_promedio_cop']

In [67]:
from sklearn.linear_model import LinearRegression

In [68]:
X_train = df_train.values
X_val = df_val.values
X_test = df_test.values

print(X_train.shape)
print(y_train.shape)

(2779, 11)
(2779,)


In [69]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](11,)","[-0.18,-0.01,-2.94,..., 1.58, 0.73, 1.04]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,381.6
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,11
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,11
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](11,)","[188.01, 19.03, 18.05,..., 14.76, 14.34, 5.88]"


In [70]:
y_pred = modelo.predict(X_val)

In [71]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)
print(f"Intercepto: {modelo.intercept_:.4f}")
print(f"Pesos: {modelo.coef_}")

Intercepto: 381.6307
Pesos: [-0.18125883 -0.01401048 -2.94201607 -2.15412263 -2.55059477 -2.43481682
 -2.39341772 -2.01839216  1.57864493  0.72527496  1.03798997]


In [72]:
mae = mean_absolute_error(y_val, y_pred)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

MAE: 1.1313
RMSE: 1.7165
R²: 0.5043


In [73]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_val)

In [74]:
mae_rf = mean_absolute_error(y_val, y_pred_rf)
rmse_rf = root_mean_squared_error(y_val, y_pred_rf)
r2_rf = r2_score(y_val, y_pred_rf)

print(mae_rf)
print(rmse_rf)
print(r2_rf)

1.1986261577398434
1.7936983476850805
0.4587319630659824


In [75]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(
    random_state=42
)

gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_val)

In [76]:
mae_gb = mean_absolute_error(y_val, y_pred_gb)
rmse_gb = root_mean_squared_error(y_val, y_pred_gb)
r2_gb = r2_score(y_val, y_pred_gb)

print(mae_gb)
print(rmse_gb)
print(r2_gb)

1.1648811259722704
1.749798616929608
0.4849021899249182


In [77]:
resultados = pd.DataFrame({
    "Modelo": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        mae,
        mae_rf,
        mae_gb
    ],
    "RMSE": [
        rmse,
        rmse_rf,
        rmse_gb
    ],
    "R2": [
        r2,
        r2_rf,
        r2_gb
    ]
})

resultados

,Modelo,MAE,RMSE,R2
0,Linear Regression,1.131274,1.716467,0.504339
1,Random Forest,1.198626,1.793698,0.458732
2,Gradient Boosting,1.164881,1.749799,0.484902


MLflow: tracking y selección del mejor modelo

In [81]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("models_factura_promedio")

modelos_mlflow = [
    {
        "nombre": "LinearRegression",
        "estimador": modelo,
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    },
    {
        "nombre": "RandomForest",
        "estimador": rf,
        "mae": float(mae_rf),
        "rmse": float(rmse_rf),
        "r2": float(r2_rf),
    },
    {
        "nombre": "GradientBoosting",
        "estimador": gb,
        "mae": float(mae_gb),
        "rmse": float(rmse_gb),
        "r2": float(r2_gb),
    },
]

for m in modelos_mlflow:
    with mlflow.start_run(run_name=m["nombre"]):
        mlflow.log_param("model", m["nombre"])
        mlflow.log_param("target_transform", "log1p")
        mlflow.log_metric("mae", m["mae"])
        mlflow.log_metric("rmse", m["rmse"])
        mlflow.log_metric("r2", m["r2"])
        mlflow.sklearn.log_model(m["estimador"], "model")

print("Runs registrados en MLflow para los 3 modelos.")

2026/06/10 09:28:11 INFO mlflow.tracking.fluent: Experiment with name 'models_factura_promedio' does not exist. Creating a new experiment.
2026/06/10 09:28:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 09:28:12 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\juegalvi\Documents\Universidad\proyectoaulaNE
2026/06/10 09:28:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/10 09:28:12 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\juegalvi\Documents\Universidad\proyectoaulaNE
2026/06/10 09:28:12 INFO mlflow.u

Runs registrados en MLflow para los 3 modelos.


Selección del mejor modelo e interpretación

In [82]:
ranking = resultados.sort_values(by=["RMSE", "MAE", "R2"], ascending=[True, True, False]).reset_index(drop=True)
mejor_modelo = ranking.iloc[0]

print("Ranking de modelos (menor RMSE y MAE, mayor R2):")
print(ranking)
print("\nModelo elegido:", mejor_modelo["Modelo"])

comentario = (
    f"Se elige {mejor_modelo['Modelo']} como mejor modelo porque obtuvo "
    f"el menor RMSE ({mejor_modelo['RMSE']:.4f}) y un MAE competitivo ({mejor_modelo['MAE']:.4f}), "
    f"además de un R2 de {mejor_modelo['R2']:.4f}. "
    "Interpretación rápida: MAE indica el error promedio absoluto en escala logarítmica del target, "
    "RMSE penaliza más los errores grandes y R2 muestra la proporción de variabilidad explicada por el modelo."
)

print("\nComentario final:")
print(comentario)

Ranking de modelos (menor RMSE y MAE, mayor R2):
              Modelo       MAE      RMSE        R2
0  Linear Regression  1.131274  1.716467  0.504339
1  Gradient Boosting  1.164881  1.749799  0.484902
2      Random Forest  1.198626  1.793698  0.458732

Modelo elegido: Linear Regression

Comentario final:
Se elige Linear Regression como mejor modelo porque obtuvo el menor RMSE (1.7165) y un MAE competitivo (1.1313), además de un R2 de 0.5043. Interpretación rápida: MAE indica el error promedio absoluto en escala logarítmica del target, RMSE penaliza más los errores grandes y R2 muestra la proporción de variabilidad explicada por el modelo.


Hiperparámetros Random Forest

In [ ]:
RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o